In [ ]:
#Data Preparation

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from docx import Document
from docx.shared import Pt, Inches

df_wide = pd.read_csv('us_data/aggregatedData.csv')

df_male = df_wide[['state', 'year',
                    'male_business_owners_psts',
                    'male_labor_force']].copy()
df_male.columns = ['state', 'year', 'business_owners_psts', 'labor_force']
df_male['gender'] = 'Male'
df_male['female'] = 0

df_female = df_wide[['state', 'year',
                      'female_business_owners_psts',
                      'female_labor_force']].copy()
df_female.columns = ['state', 'year', 'business_owners_psts', 'labor_force']
df_female['gender'] = 'Female'
df_female['female'] = 1

df = pd.concat([df_male, df_female], ignore_index=True)

df['biz_rate'] = df['business_owners_psts'] / df['labor_force']
df['log_rate'] = np.log(df['biz_rate'])

df_final = df[df['year'].between(2014, 2023)].dropna(subset=['log_rate', 'labor_force']).copy()
df_final['year'] = df_final['year'].astype(int)

In [ ]:
#Parallel Trends

sns.set_theme(style="whitegrid")

df_pre = df_final[df_final['year'] <= 2017].copy()

plt.figure(figsize=(10, 6))
sns.lineplot(data=df_final, x='year', y='log_rate', hue='gender', marker='o', linewidth=2.5)

plt.title('Parallel Trends Check: Development of Business Ownership (log Rate)', fontsize=14)
plt.ylabel('Log(Business Owners / Labor Force)')
plt.legend()
plt.savefig('parallelTrends.png', dpi=300, bbox_inches='tight')
plt.show()

model_pre = smf.ols("log_rate ~ female * C(year) + C(state)", data=df_pre)
res_pre = model_pre.fit(cov_type='cluster', cov_kwds={'groups': df_pre['state']})

print(res_pre.summary())

interaction_vars = [var for var in res_pre.params.index if 'female:C(year)' in var]
f_test = res_pre.f_test(interaction_vars)

print(f_test.summary())

doc = Document()
section = doc.sections[0]
section.page_width = Inches(11)
section.page_height = Inches(8.5)
doc.add_heading('OLS Parallel Trends Regression Results', 0)
paragraph = doc.add_paragraph()
run = paragraph.add_run(str(res_pre.summary()))
run.font.name = 'Courier New'
run.font.size = Pt(9)
doc.save("Parallel_Trends_OLS.docx")

doc = Document()
doc.add_heading('F Test Results', 0)
paragraph = doc.add_paragraph()
run = paragraph.add_run(str(f_test.summary()))
run.font.name = 'Courier New'
run.font.size = Pt(9)
doc.save("F_Test_Results.docx")